# OpenMythos — SEO / LLMO / 広告 ROI デモ

このノートブックは **3ステップ** で OpenMythos の SEO・LLMO・広告 ROI 機能を体験できます。

| ステップ | 内容 |
|----------|------|
| **Step 1** | SEO コンテンツのスコアリングと改善提案 |
| **Step 2** | LLMO（AIサーチ最適化）スコアの詳細分析と比較 |
| **Step 3** | 広告 ROI・ROAS・CPC の計算と競合分析 |

> **Colab で実行する場合:** `Runtime > Run all` を押すだけで動作します。

## インストール

In [ ]:
# Colab / 新規環境の場合のみ実行してください
# !pip install open-mythos -q

## 共通インポート

In [ ]:
import sys
import os

# リポジトリのルートから直接実行する場合のパス設定
repo_root = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from open_mythos.llmo import LLMOScorer
from open_mythos.tools_marketing import score_content, calculate_roi, search_competitor, fetch_trend

print('OpenMythos モジュール読み込み完了')

---
## Step 1: SEO コンテンツ スコアリング

`score_content()` は以下を返します:
- **LLMO トータルスコア** (0–1)
- **エンティティ密度** / **回答直接性** / **引用されやすさ**
- **キーワード密度** (ターゲット KW を指定した場合)
- **改善推奨リスト**

In [ ]:
# --- サンプルコンテンツ (Before: 改善前) ---
content_before = """
デジタルマーケティングはとても重要です。
多くの企業がインターネットを使っています。
広告もよく使われます。SNSも使われます。
これからも大切になっていくでしょう。
"""

# --- サンプルコンテンツ (After: 改善後) ---
content_after = """
デジタルマーケティングとは、Google・Meta・LINE などのデジタルプラットフォームを活用して顧客を獲得・育成する手法です。
2024年の調査によると、日本企業の SEO 投資は前年比 32% 増加し、平均 ROAS は 3.8x を記録しています。

主要チャネル比較:
- SEO: 長期的なオーガニック流入。CPL は広告の 1/5 程度
- リスティング広告: 即効性高。平均 CTR 3.5%、CPC 150–500円
- SNS広告 (Instagram/TikTok): 視覚訴求。エンゲージメント率 2–8%

出典: デジタルマーケティング白書 2024（株式会社 DI）
"""

target_kw = "デジタルマーケティング"

result_before = score_content(content_before, target_keyword=target_kw)
result_after  = score_content(content_after,  target_keyword=target_kw)

print('=' * 50)
print('【改善前】')
print(f"  LLMO トータル : {result_before['llmo_total']:.3f}")
print(f"  エンティティ密度 : {result_before['entity_density']:.3f}")
print(f"  回答直接性     : {result_before['answer_directness']:.3f}")
print(f"  引用されやすさ : {result_before['citability']:.3f}")
print(f"  改善推奨: {result_before['recommendations']}")

print()
print('【改善後】')
print(f"  LLMO トータル : {result_after['llmo_total']:.3f}")
print(f"  エンティティ密度 : {result_after['entity_density']:.3f}")
print(f"  回答直接性     : {result_after['answer_directness']:.3f}")
print(f"  引用されやすさ : {result_after['citability']:.3f}")
print(f"  改善推奨: {result_after['recommendations']}")
print('=' * 50)

---
## Step 2: LLMO スコア詳細分析と複数コンテンツ比較

`LLMOScorer.rank()` で複数バリアントを一括評価し、AIサーチで引用されやすいコンテンツを選定します。

In [ ]:
scorer = LLMOScorer()

variants = [
    # バリアント A: entity-rich
    """SEO（Search Engine Optimization）とは、Google・Bing などの検索エンジンで上位表示を獲得する施策です。
    Core Web Vitals (LCP < 2.5s, CLS < 0.1, FID < 100ms) の改善が 2024年以降は必須要件です。
    国内 SEO ツール市場は 2023年に 450億円規模に達し、年率 18% 成長しています。""",

    # バリアント B: answer-first
    """はい、SEO は 2024年も有効な集客手段です。
    理由は3つ: 検索意図の精度向上、E-E-A-T 評価の強化、AIサーチへの対応（LLMO）です。
    特に日本語コンテンツは競合が少なく ROI が高い傾向にあります。""",

    # バリアント C: 一般的な文章 (比較用)
    """SEOはとても重要なマーケティング手法です。
    検索エンジンで上位に表示されると多くのアクセスが集まります。
    正しい方法で取り組むことが大切です。""",
]

rankings = scorer.rank(variants)

labels = ['A: entity-rich', 'B: answer-first', 'C: 一般文章']
# rank() は元インデックス順ではなくスコア降順で返るため、元のインデックスを復元
scores = scorer.batch_score(variants)

print('LLMO スコア比較')
print(f"{'バリアント':<20} {'LLMO':>6} {'Entity':>8} {'Direct':>8} {'Cite':>8}")
print('-' * 55)
for label, s in zip(labels, scores):
    print(f"{label:<20} {s.llmo_total:>6.3f} {s.entity_density:>8.3f} {s.answer_directness:>8.3f} {s.citability:>8.3f}")

print()
print('ランキング (LLMO 降順):')
for pos, total, s in rankings:
    # batch_scores の index を特定
    idx = scores.index(s)
    print(f"  {pos}位: {labels[idx]}  (スコア: {total:.3f})")

In [ ]:
# compare() で Before/After の改善幅を定量評価
comparison = scorer.compare(content_before, content_after)

print('Before → After 改善比較')
print(f"  改善前スコア   : {comparison['baseline_total']:.3f}")
print(f"  改善後スコア   : {comparison['candidate_total']:.3f}")
print(f"  スコア差 (Δ)  : {comparison['delta']:+.3f}")
print(f"  改善率         : {comparison['improvement_pct']:+.1f}%")
print(f"  Entity Δ      : {comparison['entity_delta']:+.3f}")
print(f"  Directness Δ  : {comparison['directness_delta']:+.3f}")
print(f"  Citability Δ  : {comparison['citability_delta']:+.3f}")

---
## Step 3: 広告 ROI 計算と競合分析

### 3-1. ROI / ROAS / CPC 計算

In [ ]:
# シナリオ: 月次 SEO コンテンツ投資の ROI 試算
roi = calculate_roi(
    ad_spend=500_000,     # 広告費 50万円
    revenue=1_900_000,    # 売上 190万円
    cogs=400_000,         # 原価 40万円
    clicks=3_800,         # クリック数
    impressions=120_000,  # インプレッション数
)

print('広告 ROI 試算結果')
print(f"  広告費         : ¥{roi['ad_spend_usd']:,.0f}")
print(f"  売上           : ¥{roi['revenue_usd']:,.0f}")
print(f"  粗利           : ¥{roi['gross_profit_usd']:,.0f}")
print(f"  ROI            : {roi['roi_pct']:+.1f}%")
print(f"  ROAS           : {roi['roas']:.2f}x")
print(f"  CPC            : ¥{roi['cpc_usd']:.0f}")
print(f"  CTR            : {roi['ctr']:.2%}")
print(f"  CPM            : ¥{roi['cpm_usd']:.0f}")
print(f"  収益性         : {'✅ 黒字' if roi['profitable'] else '❌ 赤字'}")

### 3-2. 競合分析

In [ ]:
competitors = ["Jasper AI", "Copy.ai", "Notion AI"]

print('競合分析 (過去30日間)')
print(f"{'企業':<15} {'広告費 (USD)':>14} {'CTR':>8} {'SEO スコア':>12} {'市場シェア':>12}")
print('-' * 65)
for comp in competitors:
    data = search_competitor(comp, metric='all', period='last_30_days')
    print(
        f"{data['company']:<15}"
        f" ${data['ad_spend_usd']:>13,}"
        f" {data['avg_ctr']:>8.2%}"
        f" {data['seo_score']:>11}/100"
        f" {data['market_share_pct']:>11.1f}%"
    )

### 3-3. キーワードトレンド分析

In [ ]:
keywords = ["LLMO", "SEO最適化", "AI広告", "デジタルマーケティング"]

print('キーワード トレンド (JP)')
print(f"{'キーワード':<22} {'トレンド':>9} {'LLMO人気度':>12} {'月間検索数':>12} {'前年比':>8}")
print('-' * 68)
for kw in keywords:
    t = fetch_trend(kw, region='JP')
    rising = '↑' if t['is_rising'] else '→'
    print(
        f"{t['keyword']:<22}"
        f" {t['trend_score']:>7}/100 {rising}"
        f" {t['llmo_popularity']:>12.3f}"
        f" {t['search_volume_est']:>12,}"
        f" {t['yoy_change_pct']:>+7.1f}%"
    )

---
## まとめ

このデモで示した OpenMythos の SEO/LLMO/広告 コア機能:

| 機能 | モジュール | 主なユースケース |
|------|-----------|------------------|
| LLMO スコアリング | `open_mythos.llmo` | AIサーチ向けコンテンツ品質評価 |
| SEO コンテンツ分析 | `open_mythos.tools_marketing` | コンテンツ改善・KW密度最適化 |
| 広告 ROI 計算 | `open_mythos.tools_marketing` | キャンペーン収益性試算 |
| 競合分析 | `open_mythos.tools_marketing` | 市場ポジショニング |
| トレンド分析 | `open_mythos.tools_marketing` | コンテンツ戦略立案 |

### 次のステップ

```python
# ReActAgent で SEO ワークフローを自動化
from open_mythos.react import ReActAgent

# SwarmOrchestrator でマルチエージェント広告プランナー
from open_mythos.swarm import SwarmOrchestrator

# REST API サーバーの起動
# uvicorn open_mythos.serve.api:app --host 0.0.0.0 --port 8000
```

詳細は [GitHub README](https://github.com/hiroshi57/OpenMythos) を参照してください。